# Validity & scope experiments

Targets the two soft spots (first-token vs full-answer; the suspiciously-clean prefix-unique result)
plus uncertainty and calibration symmetry.
- **V1 Continuation decomposition (A/B/C/D)**: patch the answer direction at the FIRST step only, greedy
  continue, and classify — (A) first recovered + full alias completes, (B) first recovered then drifts,
  (C) first recovered, alias normalization mismatch, (D) weak prefix shared with a decoy. If B dominates,
  the paper's scope stays first-token selection.
- **V2 Multi-token sequence read/write** (1-3 token answers): teacher-forced sequence read vs greedy
  continuation write. Directly addresses the first-token validity concern.
- **V3 Stricter prefix-unique sweep**: unique among decoys, unique first-2 tokens, long-enough first
  token; dissociation under each with bootstrap CIs. Verifies frac_prefix_unique=1.0 is not an artifact.
- **V4 Single-token subset with bootstrap CIs**: rho_single, hard-readable fraction, not-top rate,
  answer-up recovery — all with CIs, since the subset is small.
- **V7 Symmetric alternative-down calibration**: set the competitor support DOWN to the success-level
  alternative support, making answer-up and alternative-down comparable.
- **V5 Tuned-lens scaffold**: structure only — REQUIRES fitting affine probes per layer. Logit-lens is
  used by default; flip `RUN_TUNED_LENS` once probes are trained.

Needs `rw_core.py`, `mech_core.py`, `mech_runner.py`. Run their tests first.

## 0. Config + data

In [ ]:
import numpy as np, json, gc, torch, re
import rw_core as rw, mech_core as mc, mech_runner as mr
from datasets import load_dataset
from collections import defaultdict
import pandas as pd

DEVICE="cuda" if torch.cuda.is_available() else "cpu"
DTYPE=torch.float16 if DEVICE=="cuda" else torch.float32
QA="Answer with a short factual answer.\nQuestion: {q}\nAnswer:"
TEMPLATES=[QA,"Q: {q}\nA:","Please answer concisely.\n{q}\nAnswer:","{q} The answer is"]
N_ITEMS=300
CAPS=dict(cont=120, seq=200, prefix=400, single=400, altdown=200)
RUN_TUNED_LENS=False   # requires trained affine probes; see section 7

MODELS=[
 "meta-llama/Llama-3.1-8B","meta-llama/Llama-3.2-3B","meta-llama/Llama-3.2-1B",
 "meta-llama/Llama-3.2-3B-Instruct","Qwen/Qwen2.5-3B","Qwen/Qwen2.5-3B-Instruct",
 "Qwen/Qwen2.5-7B","mistralai/Mistral-7B-v0.1",
]

ds=load_dataset("akariasai/PopQA",split="test")
def aliases(r):
    a=r["possible_answers"]
    if isinstance(a,str):
        try: a=json.loads(a)
        except: a=[a]
    return a
ITEMS=[{"q":str(r["question"]),"gold":aliases(r),"rel":str(r.get("prop","na"))} for r in ds]
REL=defaultdict(list)
for it in ITEMS:
    for a in it["gold"]: REL[it["rel"]].append(a)
np.random.default_rng(0).shuffle(ITEMS); ITEMS=ITEMS[:N_ITEMS]
print("items per model:",len(ITEMS))

## 1. Run V1/V2/V3/V4/V7 on all models (per-experiment isolation)

In [ ]:
VALID={}
for name in MODELS:
    print("="*70); print(name,flush=True)
    try:
        ctx=mr.make_ctx(name,DEVICE,DTYPE,ITEMS,REL,QA,TEMPLATES)
        out={}
        for key,fnc in [
            ("V1_cont",   lambda: mr.exp_continuation_decomposition(ctx, max_items=CAPS["cont"])),
            ("V2_seq",    lambda: mr.exp_sequence_readwrite(ctx, max_items=CAPS["seq"])),
            ("V3_prefix", lambda: mr.exp_prefix_unique_strict(ctx, max_items=CAPS["prefix"])),
            ("V4_single", lambda: mr.exp_single_token_ci(ctx, max_items=CAPS["single"])),
            ("V7_altdown",lambda: mr.exp_symmetric_altdown(ctx, max_items=CAPS["altdown"])),
        ]:
            try: out[key]=fnc(); print(f"  {key} done",flush=True)
            except Exception as e:
                import traceback; traceback.print_exc(); out[key]={"status":"error","error":f"{type(e).__name__}: {e}"}
                print(f"  {key} FAILED: {type(e).__name__}: {str(e)[:140]}",flush=True)
        VALID[name]=out
    except Exception as e:
        import traceback; traceback.print_exc(); VALID[name]={"status":"error","error":f"{type(e).__name__}: {e}"}
    finally:
        try: mr.free_ctx(ctx); del ctx
        except Exception: pass
        torch.cuda.empty_cache(); gc.collect()
json.dump(VALID,open("validity_experiments.json","w"),indent=2,default=float)
print("\nsucceeded:",len([k for k,v in VALID.items() if "status" not in v]),"/",len(VALID))

## 2. V1 — continuation decomposition (A/B/C/D)
A = first recovered + full alias completes; B = first recovered then drifts; C = alias normalization
mismatch; D = weak prefix shared with a decoy. If B dominates, scope stays first-token selection.

In [ ]:
rows=[{"model":nm.split("/")[-1],"n":v["V1_cont"]["n"],
        "A_full":round(v["V1_cont"]["A_first_and_full"],3),
        "B_drift":round(v["V1_cont"]["B_first_then_drift"],3),
        "C_alias":round(v["V1_cont"]["C_alias_mismatch"],3),
        "D_weakprefix":round(v["V1_cont"]["D_weak_prefix"],3),
        "first_not_recov":round(v["V1_cont"]["first_not_recovered"],3),
        "tf_rest_lp":round(v["V1_cont"]["tf_rest_mean_logprob"],2)}
       for nm,v in VALID.items() if "status" not in v and "status" not in v.get("V1_cont",{})]
print(pd.DataFrame(rows).to_string(index=False))
print("\nif B (drift) dominates among recovered-first cases, report the paper as first-token SELECTION.")
print("if A is competitive, continuation often completes -> the claim extends to short answers.")

## 3. V2 — multi-token sequence read/write (1-3 token answers)

In [ ]:
rows=[{"model":nm.split("/")[-1],"n":v["V2_seq"]["n"],
        "rho_seqread_write":round(v["V2_seq"]["rho_seqread_negwrite"],3),
        "gen_correct":round(v["V2_seq"]["gen_correct_rate"],3),
        "mean_seq_read":round(v["V2_seq"]["mean_seq_read"],3)}
       for nm,v in VALID.items() if "status" not in v and "status" not in v.get("V2_seq",{})]
print(pd.DataFrame(rows).to_string(index=False))
print("\npositive rho on short multi-token answers shows the dissociation is not purely a first-token effect.")

## 4. V3 — stricter prefix-unique sweep (verify frac=1.0 is real)
Fractions under stricter criteria and the dissociation (bootstrap CI) restricted to each.

In [ ]:
def fmt(ci): return f"{ci['point']:.3f} [{ci['lo']:.3f},{ci['hi']:.3f}] (n{ci['n']})"
rows=[{"model":nm.split("/")[-1],"n":v["V3_prefix"]["n"],
        "frac_u1_decoy":round(v["V3_prefix"]["frac_unique1_decoy"],3),
        "frac_u2":round(v["V3_prefix"]["frac_unique2"],3),
        "frac_long":round(v["V3_prefix"]["frac_long_enough"],3),
        "frac_strict":round(v["V3_prefix"]["frac_strict"],3),
        "rho_all":fmt(v["V3_prefix"]["rho_all"]),
        "rho_strict":fmt(v["V3_prefix"]["rho_strict"])}
       for nm,v in VALID.items() if "status" not in v and "status" not in v.get("V3_prefix",{})]
print(pd.DataFrame(rows).to_string(index=False))
print("\nif frac_long / frac_strict drop well below 1.0 but rho_strict stays close to rho_all, the")
print("dissociation survives strict prefix-uniqueness (the 1.0 was driven by short/space tokens).")

## 5. V4 — single-token subset metrics with bootstrap CIs

In [ ]:
def fmt(ci): return f"{ci['point']:.3f} [{ci['lo']:.3f},{ci['hi']:.3f}]"
rows=[{"model":nm.split("/")[-1],"n":v["V4_single"]["n"],"n_single":v["V4_single"]["n_single"],
        "rho_all":fmt(v["V4_single"]["rho_all"]),"rho_single":fmt(v["V4_single"]["rho_single"]),
        "hard_readable":fmt(v["V4_single"]["hard_readable_frac"]),
        "nottop":fmt(v["V4_single"]["nottop_rate"]),
        "answer_up_recov":fmt(v["V4_single"]["answer_up_recovery"])}
       for nm,v in VALID.items() if "status" not in v and "status" not in v.get("V4_single",{})]
print(pd.DataFrame(rows).to_string(index=False))
print("\nwide CIs on rho_single (small n_single) are reported honestly; the point estimate still helps.")

## 6. V7 — symmetric alternative-down calibration
Answer-up to success-level vs competitor-down to success-level alternative support (comparable scales),
and both. Removes the asymmetry between the two interventions.

In [ ]:
rows=[{"model":nm.split("/")[-1],"n":v["V7_altdown"]["n"],
        "T_answer":round(v["V7_altdown"]["target_answer"],2),
        "T_alt_success":round(v["V7_altdown"]["target_alt_success_level"],2),
        "answer_up":round(v["V7_altdown"]["answer_up_recovery"],3),
        "alt_down_sym":round(v["V7_altdown"]["alt_down_to_success_recovery"],3),
        "both":round(v["V7_altdown"]["both_recovery"],3)}
       for nm,v in VALID.items() if "status" not in v and "status" not in v.get("V7_altdown",{})]
print(pd.DataFrame(rows).to_string(index=False))
print("\neven with a symmetric (success-calibrated) competitor-down target, answer-up should dominate;")
print("this answers the 'you made the interventions asymmetric' objection.")

## 7. V5 — tuned-lens scaffold (NOT a drop-in; requires trained probes)
The logit lens reads `LN_f(h_l) @ W_U`. A tuned lens replaces this with a per-layer affine probe
`A_l h_l + b_l` fit to match the final logits. This scaffold shows where the probe plugs in; it does
NOT fit the probes. To use it: train `A_l, b_l` per layer (e.g. on held-out activations, minimizing KL
to final logits), save them, load below, and set `RUN_TUNED_LENS=True`. Then rerun the dissociation
with the tuned read in place of `r_int`. Honesty: until probes are fit, this is a placeholder, and the
paper should hedge the read as 'logit-lens decodability under this probe.'

In [ ]:
if RUN_TUNED_LENS:
    # EXPECTS: tuned_probes[model][layer] = (A_l (d->V or d->d then W_U), b_l). Fit elsewhere.
    print("Tuned-lens path is a scaffold. Load your trained probes here and recompute r_int as")
    print("log_softmax(A_l h_l + b_l). Then call the dissociation runners with the tuned read.")
    print("NOT IMPLEMENTED inline -- requires the fitted affine maps.")
else:
    print("RUN_TUNED_LENS=False. Using logit-lens read (default). The read-definition sweep (S3) and")
    print("hard-decoy headline (P6) are the available robustness evidence; hedge read as")
    print("'decodability under the logit lens' in the paper. A tuned lens on Llama-3.1-8B and")
    print("Qwen2.5-3B would be the cleanest faithfulness defense if a reviewer requires it.")

## Notes
- **V1** is the scope-decider: B-dominant => keep the paper at first-token selection (clean and
  honest); A-competitive => you can extend to short answers. Either way it resolves the P8 ambiguity.
- **V2** complements V1 with a teacher-forced multi-token read; a positive rho on 1-3 token answers
  shows it is not purely first-token.
- **V3** verifies the prefix-unique result: expect frac_long < 1.0 (some short/space first tokens), and
  the dissociation should survive the strict subset.
- **V4** attaches CIs so the small single-token subset is reported honestly.
- **V7** makes the two interventions comparable; answer-up should still dominate.
- **V5** tuned lens is the one outstanding faithfulness check; it needs trained probes and is scoped as
  a separate task. Until then, hedge the read.
- Raise `N_ITEMS`/`CAPS` for final numbers.